# AHP-Experiment: Auswertung

Mehrere Ergebnisdateien im Vergleich: LLM-Laeufe aus `experiment.py` und die
Probandendaten aus `transform_interviews.py`. Die Dateien stehen in `FILES`, die
erste ist die Referenz fuer die Abweichungstabelle.
Gewichte und CR rechnet pyDecision (`ahp_method`), das Notebook baut nur die Matrizen.

In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from pyDecision.algorithm import ahp_method        # pip install pyDecision

# Beliebig viele Ergebnisdateien; die erste ist die Referenz fuer den Vergleich.
FILES = [
    Path("../results/InterviewResults/data_ahpsupplierselectionjs23_2026-09-19_12-33_clean.json"),
    Path("../results/20260924-101539_openai_gpt-5.4-mini-2026-03-17_stateless.json"),
    Path("../results/20260924-101309_anthropic_claude-haiku-4-5-20251001_stateless.json"),
    Path("../results/20260924-102357_mistral_mistral-small-2603_stateless.json"),
]
FIGDIR = Path("figures"); FIGDIR.mkdir(exist_ok=True)

CRIT = ["liefertreue", "flexibilitaet", "kosten"]      # feste Reihenfolge = Matrixachsen
# Beschriftungen auf Englisch wie in der Arbeit (CRIT_LABEL/CASE_LABEL in
# build_results_tex.py); sie stehen in Abbildungen und in der Spalte ranking.
LABEL = {"liefertreue": "Delivery reliability", "flexibilitaet": "Flexibility", "kosten": "Cost"}
CASE_LABEL = {"ketten": "Chains", "batteriepacks": "Battery packs",
              "konnektivitaetsmodule": "Connectivity modules"}
# Gewichtungsverfahren in ahp_method: "g" geometrisches Mittel, "me" Eigenvektor
# (bei 3x3-Matrizen identisch), "m" Mittel der normierten Spalten. CR jeweils
# nach Saaty mit dem Random Index von pyDecision (0,58 bei n = 3).
AHP_METHOD = "g"
# Paarvergleiche fuer den Export als Matrixeintrag a_ij, i vor j in CRIT: die
# gewaehlte Intensitaet, wenn das erste Kriterium vorgezogen wurde, sonst ihr
# Kehrwert (1 = gleich wichtig).
PAIRS = [(a, b) for i, a in enumerate(CRIT) for b in CRIT[i + 1:]]


def to_matrix(comparisons):
    """Drei Paarvergleiche -> reziproke 3x3-Matrix."""
    idx = {c: i for i, c in enumerate(CRIT)}
    A = np.ones((len(CRIT), len(CRIT)))
    for c in comparisons:
        i, j = idx[c["a"]], idx[c["b"]]
        s = max(int(c["intensity"]), 1)                 # ein Run liefert 0 statt 1
        if c["preference"] == "gleich":
            v = 1.0
        elif c["preference"] == c["a"]:
            v = float(s)
        elif c["preference"] == c["b"]:
            v = 1.0 / s
        else:
            raise ValueError(f"unbekannte preference: {c}")
        A[i, j], A[j, i] = v, 1.0 / v
    return A


def source_label(meta):
    """'Participants' oder Modell mit Bedingung."""
    if meta["condition"] == "human":
        return "Participants"
    return f'{meta["model"]} ({meta["condition"]})'


SOURCES, frames = [], []
for nr, path in enumerate(FILES, start=1):
    raw = json.loads(path.read_text(encoding="utf-8"))
    quelle = source_label(raw["meta"])
    if quelle in SOURCES:                               # gleiche Quelle mehrfach: Nummer anhaengen
        quelle = f"{quelle} #{nr}"
    SOURCES.append(quelle)
    print(f"{quelle}: {raw['summary']} | temp {raw['meta']['temperature']} | seed {raw['meta']['seed']}")

    records = []
    for r in raw["responses"]:
        # Ungueltige Antworten nicht auswerten: to_matrix wuerde fehlende
        # Vergleiche still als "gleich" behandeln.
        if not r["valid"]:
            continue
        A = to_matrix(r["comparisons"])
        w, cr = ahp_method(A, wd=AHP_METHOD)
        records.append({
            "quelle": quelle, "run": r["run"], "case": r["case_id"], "position": r["position"],
            "sequence": "-".join(s[:3] for s in r["sequence"]) if r["sequence"] else "stateless",
            **{c: w[i] for i, c in enumerate(CRIT)},
            "CR": cr,
            "temperature": r["temperature"],
            **{f"{a}:{b}": A[CRIT.index(a), CRIT.index(b)] for a, b in PAIRS},
        })
    frames.append(pd.DataFrame(records))

df = pd.concat(frames, ignore_index=True)
df["ranking"] = df[CRIT].apply(
    lambda r: " > ".join(LABEL[c] for c in r.sort_values(ascending=False).index), axis=1)
df["quelle"] = pd.Categorical(df["quelle"], categories=SOURCES, ordered=True)
df["case"] = pd.Categorical(df["case"], categories=list(CASE_LABEL), ordered=True)
df = df.sort_values(["quelle", "case", "run"]).reset_index(drop=True)
df.head()

Probanden: {'total': 42, 'valid': 42, 'invalid': 0} | temp None | seed None
gpt-5.4-mini-2026-03-17 (stateless): {'total': 300, 'valid': 300, 'invalid': 0} | temp 0.0 | seed None
claude-haiku-4-5-20251001 (stateless): {'total': 300, 'valid': 300, 'invalid': 0} | temp 0.0 | seed None
mistral-small-2603 (stateless): {'total': 300, 'valid': 300, 'invalid': 0} | temp 0.0 | seed None


,quelle,run,case,position,sequence,liefertreue,flexibilitaet,kosten,CR,ranking
0,Probanden,160,ketten,1,bat-ket-kon,0.222518,0.650648,0.126834,0.254120,Flexibilitaet > Liefertreue > Kosten
1,Probanden,164,ketten,1,kon-ket-bat,0.751112,0.043628,0.205261,0.375594,Liefertreue > Kosten > Flexibilitaet
2,Probanden,174,ketten,2,bat-kon-ket,0.796828,0.051776,0.151395,0.254120,Liefertreue > Kosten > Flexibilitaet
3,Probanden,175,ketten,1,kon-ket-bat,0.628196,0.086427,0.285377,0.316760,Liefertreue > Kosten > Flexibilitaet
4,Probanden,185,ketten,0,ket-bat-kon,0.679480,0.064374,0.256146,0.147475,Liefertreue > Kosten > Flexibilitaet


## Kennzahlen

Je Fall und Quelle; `n` ist die Zahl der Runs bzw. Teilnahmen.

In [13]:
by = ["case", "quelle"]
n = df.groupby(by, observed=True).size().rename("n")
mean_w = df.groupby(by, observed=True)[CRIT].mean().rename(columns=LABEL).join(n)
cr_tab = df.groupby(by, observed=True)["CR"].agg(
    mean="mean", median="median", max="max",
    anteil_ueber_0_1=lambda s: (s > 0.1).mean())
patterns = df.groupby(by, observed=True).apply(
    lambda g: g[CRIT].round(6).apply(tuple, axis=1).nunique(), include_groups=False)

print("Mittlere Gewichte\n", mean_w.round(3), "\n")
print("Konsistenz\n", cr_tab.round(3), "\n")
print("Verschiedene Antwortmuster je Fall\n", patterns.to_string())

Mittlere Gewichte
                                                              Liefertreue  \
case                  quelle                                               
ketten                Probanden                                    0.532   
                      gpt-5.4-mini-2026-03-17 (stateless)          0.649   
                      claude-haiku-4-5-20251001 (stateless)        0.648   
                      mistral-small-2603 (stateless)               0.690   
batteriepacks         Probanden                                    0.165   
                      gpt-5.4-mini-2026-03-17 (stateless)          0.091   
                      claude-haiku-4-5-20251001 (stateless)        0.106   
                      mistral-small-2603 (stateless)               0.170   
konnektivitaetsmodule Probanden                                    0.235   
                      gpt-5.4-mini-2026-03-17 (stateless)          0.188   
                      claude-haiku-4-5-20251001 (stateless)        0.

## Vergleich mit der Referenz

Abweichung der mittleren Gewichte von der ersten Datei in `FILES` (positiv =
hoeher gewichtet als die Referenz) und die haeufigste Rangfolge je Quelle.

In [9]:
REF = SOURCES[0]
mean_all = df.groupby(["case", "quelle"], observed=True)[CRIT].mean()
diff = (mean_all.sub(mean_all.xs(REF, level="quelle"), level="case")
                .drop(REF, level="quelle").rename(columns=LABEL))
print(f"Abweichung der mittleren Gewichte von: {REF}\n", diff.round(3), "\n")


def top_ranking(s):
    anteil = s.value_counts(normalize=True)
    return f"{anteil.index[0]} ({anteil.iloc[0]:.0%})"


print("Haeufigste Rangfolge (Anteil)")
df.groupby(["case", "quelle"], observed=True)["ranking"].agg(top_ranking).unstack("quelle")

Abweichung der mittleren Gewichte von: Probanden
                                                              Liefertreue  \
case                  quelle                                               
ketten                gpt-5.4-mini-2026-03-17 (stateless)          0.117   
                      claude-haiku-4-5-20251001 (stateless)        0.117   
                      mistral-small-2603 (stateless)               0.159   
batteriepacks         gpt-5.4-mini-2026-03-17 (stateless)         -0.074   
                      claude-haiku-4-5-20251001 (stateless)       -0.058   
                      mistral-small-2603 (stateless)               0.006   
konnektivitaetsmodule gpt-5.4-mini-2026-03-17 (stateless)         -0.047   
                      claude-haiku-4-5-20251001 (stateless)       -0.036   
                      mistral-small-2603 (stateless)              -0.052   

                                                             Flexibilitaet  \
case                  quelle       

quelle,Probanden,gpt-5.4-mini-2026-03-17 (stateless),claude-haiku-4-5-20251001 (stateless),mistral-small-2603 (stateless)
case,,,,
ketten,Liefertreue > Kosten > Flexibilitaet (71%),Liefertreue > Kosten > Flexibilitaet (84%),Liefertreue > Kosten > Flexibilitaet (100%),Liefertreue > Kosten > Flexibilitaet (100%)
batteriepacks,Flexibilitaet > Kosten > Liefertreue (57%),Flexibilitaet > Kosten > Liefertreue (99%),Flexibilitaet > Kosten > Liefertreue (100%),Flexibilitaet > Liefertreue > Kosten (100%)
konnektivitaetsmodule,Flexibilitaet > Liefertreue > Kosten (57%),Flexibilitaet > Liefertreue > Kosten (100%),Flexibilitaet > Liefertreue > Kosten (100%),Flexibilitaet > Liefertreue > Kosten (98%)


## Abbildungsstil

Einmal global, danach nicht mehr anfassen. Vektorexport. Jede Quelle hat eine Farbe
und zusaetzlich einen Marker bzw. eine Schraffur, damit die Zuordnung auch in
Graustufen und bei Farbfehlsichtigkeit lesbar bleibt.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": False,
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.4,
    "legend.frameon": False,
    "lines.markersize": 4,
    "hatch.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Feste Reihenfolge je Quelle, nie zyklisch neu vergeben. Die ersten drei Farben
# sind auch bei Farbfehlsichtigkeit paarweise unterscheidbar; ab der vierten
# Quelle traegt vor allem Marker bzw. Schraffur die Zuordnung.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
HATCHES = ["", "////", "\\\\\\\\", "xxxx", "....", "----", "||||", "++++"]
if len(SOURCES) > len(COLORS):
    raise ValueError(f"Hoechstens {len(COLORS)} Dateien in FILES")
STYLE = {q: {"color": COLORS[i], "marker": MARKERS[i], "hatch": HATCHES[i]}
         for i, q in enumerate(SOURCES)}
K = len(SOURCES)
SLOT = 0.8                                  # Breite, die sich die Quellen je Kategorie teilen
OFFSET = {q: (i - (K - 1) / 2) * SLOT / K for i, q in enumerate(SOURCES)}
HALF = SLOT / K / 2                         # halbe Breite je Quelle


def legend(fig, bars=False, ncol=3, extra=()):
    """Legende ueber der Abbildung; Marker fuer Punkte, Schraffur fuer Balken.
    `extra` sind weitere Eintraege, die vor den Quellen stehen."""
    if bars:
        handles = [Patch(facecolor=STYLE[q]["color"], hatch=STYLE[q]["hatch"],
                         edgecolor="white", linewidth=0, label=q) for q in SOURCES]
    else:
        handles = [Line2D([], [], ls="", markersize=5, color=STYLE[q]["color"],
                          marker=STYLE[q]["marker"], label=q) for q in SOURCES]
    fig.legend(handles=[*extra, *handles], loc="lower center", bbox_to_anchor=(0.5, 1.0),
               ncol=min(K + len(extra), ncol))


def save(fig, name):
    for ext in ("pdf", "svg"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print("gespeichert:", name)

## Abbildung 1 — Gewichte je Fall

Eine Abbildung je Fall. Der Balken ist das mittlere Gewicht der Quelle, der
Fehlerbalken reicht vom kleinsten bis zum groessten Gewicht ueber die Runs bzw.
Teilnahmen. Der umrandete erste Balken sind die angenommenen Zielgewichte aus
`ground_truth.yaml`.

In [ ]:
import yaml

# Angenommene Zielgewichte aus ground_truth.yaml als eigener Balken vor den
# Quellen, als Umriss und ohne Fehlerbalken. Fehlt die Datei, entfaellt er.
GT_PATH = Path("ground_truth.yaml")
GT = yaml.safe_load(GT_PATH.read_text(encoding="utf-8"))["cases"] if GT_PATH.is_file() else {}
GT_LABEL = "Assumed target weights"
GT_STYLE = {"facecolor": "white", "edgecolor": "black", "linewidth": 0.8}
GAP = 0.02                                  # Luecke zwischen benachbarten Balken

for case, g in df.groupby("case", observed=True):
    reihe = ([GT_LABEL] if case in GT else []) + SOURCES
    breite = SLOT / len(reihe)
    versatz = {q: (i - (len(reihe) - 1) / 2) * breite for i, q in enumerate(reihe)}
    fig, ax = plt.subplots(figsize=(6.0, 2.6))
    x = np.arange(len(CRIT))
    if case in GT:
        ax.bar(x + versatz[GT_LABEL], [GT[case][c] for c in CRIT], width=breite - GAP,
               zorder=3, **GT_STYLE)
    for q, gq in g.groupby("quelle", observed=True):
        mittel = gq[CRIT].mean().to_numpy()
        unten, oben = gq[CRIT].min().to_numpy(), gq[CRIT].max().to_numpy()
        # Clip: bei lauter gleichen Werten liegt das Mittel durch Rundung knapp daneben.
        spanne = np.clip([mittel - unten, oben - mittel], 0, None)
        ax.bar(x + versatz[q], mittel, width=breite - GAP, color=STYLE[q]["color"],
               hatch=STYLE[q]["hatch"], edgecolor="white", linewidth=0, zorder=3)
        ax.errorbar(x + versatz[q], mittel, yerr=spanne, fmt="none",
                    ecolor="black", elinewidth=0.8, capsize=2.5, capthick=0.8, zorder=4)
    ax.set_xticks(x)
    ax.set_xticklabels([LABEL[c] for c in CRIT])
    ax.set_title(CASE_LABEL[case])
    ax.set_xlim(-0.5, len(CRIT) - 0.5)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Weight")
    ax.xaxis.grid(False)
    ziel = [Patch(label=GT_LABEL, **GT_STYLE)] if case in GT else []
    legend(fig, bars=True, ncol=2, extra=ziel)   # zwei Spalten, sonst breiter als die Achse
    save(fig, f"01_gewichte_{case}")
    plt.show()

## Abbildung 2 — Konsistenz

Logarithmische Achse, sonst erdrueckt ein einzelner hoher CR alles andere. Perfekt
konsistente Urteile (CR = 0) liegen am unteren Rand, weil 0 auf der log-Achse nicht
darstellbar ist. Die Zahl ueber jeder Spalte ist der Anteil mit CR > 0,1.

In [ ]:
rng = np.random.default_rng(0)
CR_FLOOR = 6e-3                              # CR = 0 -> unterer Rand
fig, ax = plt.subplots(figsize=(4.6, 2.8))

for i, case in enumerate(CASE_LABEL):
    for q in SOURCES:
        y = df.loc[(df["case"] == case) & (df["quelle"] == q), "CR"].to_numpy()
        if y.size == 0:
            continue
        x = i + OFFSET[q]
        ax.scatter(x + rng.uniform(-0.6, 0.6, y.size) * HALF, np.clip(y, CR_FLOOR, None),
                   s=14, alpha=0.5, color=STYLE[q]["color"], marker=STYLE[q]["marker"],
                   edgecolors="none", zorder=3)
        ax.annotate(f"{(y > 0.1).mean():.0%}", (x, 2.2), ha="center", fontsize=7)

ax.annotate("CR > 0.1:", (-0.5, 2.2), xytext=(-4, 0), textcoords="offset points",
            ha="right", fontsize=7)
ax.axhline(0.1, color="black", lw=0.9, ls="--", zorder=2)
ax.set_yscale("log")
ax.set_ylim(5e-3, 4)
ax.set_xlim(-0.5, len(CASE_LABEL) - 0.5)
ax.set_xticks(range(len(CASE_LABEL)))
ax.set_xticklabels([CASE_LABEL[c] for c in CASE_LABEL], rotation=20, ha="right")
ax.set_ylabel("Consistency Ratio")
ax.xaxis.grid(False)

sec = ax.secondary_yaxis("right")
sec.set_yticks([0.1]); sec.set_yticklabels(["0.1"])
sec.tick_params(length=0)
sec.spines["right"].set_visible(False)
sec.minorticks_off()

legend(fig)
save(fig, "02_konsistenz")
plt.show()

## Export

Ein Long-Format fuer alles Weitere, eine Zeile je Quelle, Run und Fall. Semikolon
und Dezimalkomma, damit Excel nicht meckert.

In [23]:
out = df[["quelle", "run", "case", "position", "sequence", *CRIT, "CR", "ranking",
          "temperature", *(f"{a}:{b}" for a, b in PAIRS)]]
out.to_csv("runs.csv", sep=";", decimal=",", encoding="utf-8-sig", index=False)
print(f"{len(out)} Zeilen -> runs.csv")
out.tail(3)

942 Zeilen -> runs.csv


,quelle,run,case,position,sequence,liefertreue,flexibilitaet,kosten,CR,ranking
939,mistral-small-2603 (stateless),98,konnektivitaetsmodule,None,stateless,0.184307,0.721536,0.094157,0.159566,Flexibilitaet > Liefertreue > Kosten
940,mistral-small-2603 (stateless),99,konnektivitaetsmodule,None,stateless,0.184307,0.721536,0.094157,0.159566,Flexibilitaet > Liefertreue > Kosten
941,mistral-small-2603 (stateless),100,konnektivitaetsmodule,None,stateless,0.184307,0.721536,0.094157,0.159566,Flexibilitaet > Liefertreue > Kosten
